In [1]:
from router_patching import *
from router_injection import *

/home/dylan/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
# Set seeds
seed = 42
np.random.seed(seed);
torch.manual_seed(seed);

In [ ]:
# "Patch" a model, replacing router weights such that only first k experts are ever selected
model_choice = "deepseek_bitsandbytes"
patched_dir = "./patched_" + model_choice

# If patched model does not already exist, adjust weights.
if not os.path.isdir(patched_dir):
    model, tokenizer = patch_routers(model_choice, patched_dir)
else: # Otherwise just load the model
    model = AutoModelForCausalLM.from_pretrained(patched_dir)
    tokenizer = AutoTokenizer.from_pretrained(patched_dir)

config.json: 0.00B [00:00, ?B/s]

`rope_scaling`'s factor field must be a float >= 1, got 40
`rope_scaling`'s beta_fast field must be a float, got 32
`rope_scaling`'s beta_slow field must be a float, got 1


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-000004.safetensors:   0%|          | 0.00/5.64G [00:00<?, ?B/s]

model-00003-of-000004.safetensors:   0%|          | 0.00/8.59G [00:00<?, ?B/s]

model-00001-of-000004.safetensors:   0%|          | 0.00/8.59G [00:00<?, ?B/s]

model-00002-of-000004.safetensors:   0%|          | 0.00/8.59G [00:00<?, ?B/s]

In [ ]:
# Attach monitoring probe
if model_choice in ["qwen", "qwen_bitsandbytes", "qwen_gptq"]:
    probe = MoEProbeQwen(model)
elif model_choice in ["deepseek", "deepseek_bitsandbytes"]:
    probe = MoEProbeDeepSeek(model)
elif model_choice in ["mistral", "mistral_bitsandbytes"]:
    probe = MoEProbeMistral(model)

In [ ]:
# Now do some inference and observe routing behavior.
prompt = "Explain in detail the political system of the People's Republic of China."

# Start timing
start_time = time.time()
    
# Generate response and get metrics
response, probs, active_experts = single_generate(model, tokenizer, probe,
                                                  prompt=prompt, max_new_tokens=100)

# Stop timing
end_time = time.time()
inference_time = end_time - start_time
print(f"Inference took {inference_time:.3f}s.")

print(response)

In [ ]:
probe.plot_loadbalance(router_id=1)